# Learning Log — Part B: The Researcher's Playground
## Avenue 3: Advanced NLP Architecture — Bidirectional LSTM
**Student:** Rajmohan Karthikeyan

---

This Learning Log documents every new concept I independently researched and implemented in Part B that was **not covered in lecture**. Each section follows the required format:
- **The Concept** — what it is and why I chose it
- **The Implementation** — the actual code I used
- **The Learning** — how it works under the hood and its impact

---
## What I Already Know — From Lectures

Before documenting what is new, here is everything I already covered in class that is reused in Part B:

| Topic | Lab |
|---|---|
| Lowercasing, punctuation removal, whitespace handling | Lab 3a |
| Stop word removal, contractions lookup table | Lab 3a |
| Stemming with `PorterStemmer` | Lab 3c |
| Lemmatization with `WordNetLemmatizer` | Lab 3c |
| Loading `.txt` files into Pandas DataFrames | Lab 2a |
| Exploring DataFrames — `.head()`, `.value_counts()`, `.dtypes` | Lab 2a |
| CountVectorizer — Bag of Words | Lab 4b |
| TF-IDF Vectorizer | Lab 4b |
| `fit_transform` on train only, `transform` on val/test | Lab 5a |
| Building a Sequential Dense neural network | Lab 5a |
| Compiling with `adam` + `categorical_crossentropy` | Lab 5a |
| Plotting Loss and Accuracy curves | Lab 5a |
| Confusion Matrix and Classification Report | Lab 5a |
| Predicting on new sentences | Lab 5a |

Everything below this section is what I researched independently for Part B.

---
## Why Part A Had a Ceiling — The Problem I Needed to Solve

Before explaining what is new, it is important to understand why the Part A baseline failed.

### What is TF-IDF and why does it have a ceiling?

**TF-IDF (Term Frequency — Inverse Document Frequency)** converts words into numbers based on two things:
- **Term Frequency (TF)** — how often a word appears in a sentence
- **Inverse Document Frequency (IDF)** — how rare the word is across all sentences. Common words like "I", "am", "the" get low scores. Unique emotional words like "terrified", "furious", "joyful" get high scores.

The core problem is that TF-IDF only cares about **what** words appear, not **where** they appear. It turns every sentence into a flat bag of scores with no order:

```
"I am not happy"   →  [0.0, 0.21, 0.37, 0.18, ...]
"I am happy not"   →  [0.0, 0.21, 0.37, 0.18, ...]
```

Both sentences produce the **exact same vector** even though they mean different things. The word "not" has no connection to "happy" — they are just two separate numbers.

### What is a Dense layer and why does it have a ceiling?

A **Dense layer** is the most basic neural network layer. Every neuron connects to every other neuron in the next layer. Each connection has a **weight** — a number the model learns during training that says how important that connection is.

The problem is Dense takes all inputs at once as a flat list. It has no concept of order — it does not know word 3 came before word 4. Since TF-IDF already destroyed the order, and Dense cannot recover it, the two problems stack:

1. TF-IDF destroys word order before the model even sees the data
2. Dense has no way to recover that lost information

This is why Part A hit its ceiling — the model learned everything it could from word frequencies in the first few epochs, then had nothing left to learn except noise, causing it to memorise the training data instead of generalising.

---
## New Concept 1 — Tokenizer and Padding

### The Concept

In Part A, TF-IDF handled vectorization outside the model. In Part B, I replaced this with a **Tokenizer + pad_sequences** pipeline that preserves word order.

**Why I chose it:** The Bidirectional LSTM requires sequential input — each word must be a separate integer at a specific position. TF-IDF collapses all words into one flat vector, making it incompatible with LSTM. The Tokenizer solves this.

### The Implementation

```python
VOCAB_SIZE = 10000
MAX_LEN    = 50
EMBED_DIM  = 64

# Build vocabulary on TRAIN only — same rule as TF-IDF fit_transform
tokenizer = Tokenizer(num_words=10000, oov_token='<OOV>')
tokenizer.fit_on_texts(train_df['clean'])

# Convert text → integer sequences (transform only for val/test)
X_train_seq = tokenizer.texts_to_sequences(train_df['clean'])
X_val_seq   = tokenizer.texts_to_sequences(val_df['clean'])
X_test_seq  = tokenizer.texts_to_sequences(test_df['clean'])

# Pad all sequences to the same fixed length
X_train = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding='post', truncating='post')
X_val   = pad_sequences(X_val_seq,   maxlen=MAX_LEN, padding='post', truncating='post')
X_test  = pad_sequences(X_test_seq,  maxlen=MAX_LEN, padding='post', truncating='post')
```

### The Learning — How It Works Under the Hood

**The Tokenizer** builds a dictionary that maps every unique word in the training data to a unique integer:
- `"feel"` → `4`
- `"happy"` → `27`
- `"angry"` → `183`

The sentence `"i feel happy"` becomes `[4, 27]` — a list of integers that **preserves word order**.

**`num_words=10000`** — only the 10,000 most common words are kept. Any word outside this is treated as unknown.

**`oov_token='<OOV>'`** — Out Of Vocabulary. When the model sees a word in val/test that was never in training, instead of crashing it maps that word to a special placeholder token at index 1. This handles unknown words gracefully.

**Critical rule — same as TF-IDF:** The Tokenizer is only `fit_on_texts` on training data. Val and Test only use `texts_to_sequences` — never fit again. Fitting on test data would be data leakage.

**`pad_sequences`** — after tokenizing, every sentence is a different length. The BiLSTM needs all inputs to be exactly the same size. `pad_sequences` fixes this:
- `padding='post'` — zeros added to the END of short sentences
- `truncating='post'` — sentences longer than 50 words are cut from the end
- Result: every sentence becomes exactly 50 integers

```
"i feel angry"  →  [4, 183, 0, 0, 0, ..., 0]   ← 47 zeros added at the end
```

**Impact vs Part A:**

| | Part A (TF-IDF) | Part B (Tokenizer) |
|---|---|---|
| Word order preserved | ❌ No | ✅ Yes |
| Output format | Flat sparse vector (5,000 numbers) | Integer sequence (50 numbers) |
| Compatible with LSTM | ❌ No | ✅ Yes |
| Fit on train only | ✅ Yes | ✅ Yes |

---
## New Concept 2 — Embedding Layer

### The Concept

The Embedding layer is the first layer inside the Part B model. It is the **main vectorization technique** in Part B, replacing TF-IDF entirely.

**Why I chose it:** TF-IDF gives every word a single fixed frequency score that never changes and carries no meaning. An Embedding layer gives every word a learned 64-dimensional vector that captures semantic relationships — so "happy" and "joyful" end up similar, while "angry" and "happy" end up different.

### The Implementation

```python
Embedding(input_dim=VOCAB_SIZE, output_dim=EMBED_DIM, input_length=MAX_LEN)
# Embedding(input_dim=10000, output_dim=64, input_length=50)
```

### The Learning — How It Works Under the Hood

An Embedding layer is a **lookup table** — a matrix of shape **(10,000 × 64)**:
- 10,000 rows — one row per word in the vocabulary
- 64 columns — the embedding vector for that word

When the model receives integer `27` for "happy", it simply **looks up row 27** in the matrix and returns the 64 numbers stored there:
```
Word index 27 → look up row 27 → [0.23, -0.11, 0.87, 0.04, -0.56, ...]
                                    ↑ these 64 numbers represent "happy"
```

**`input_dim=10000`** — vocabulary size. The table has 10,000 rows, one per word.

**`output_dim=64`** — each word gets a vector of 64 numbers. This is a hyperparameter. With 10,000 words, 64 dimensions is appropriate. Larger vocabularies (e.g. 50,000 words) would use 128 or 256 dimensions.

**`input_length=50`** — the padded sequence length, so the layer knows what to expect.

**How the weights are learned:** At the start of training, all 10,000 × 64 = 640,000 values are randomly initialised. During training, backpropagation sends gradients back through the BiLSTM all the way to the Embedding layer. The model adjusts the embedding values so that words appearing in similar emotional contexts get pushed closer together in the 64-dimensional space:

```
After training:
"happy"  vector → [0.67,  0.34,  0.12, ...]
"joyful" vector → [0.71,  0.29,  0.15, ...]   ← very similar to "happy"
"angry"  vector → [-0.34, 0.67, -0.23, ...]   ← very different from "happy"
```

**What the 64 dimensions represent:** The model figures out what to encode in each dimension entirely on its own. In practice, dimensions tend to capture concepts like sentiment, intensity, or emotional category — but the model learns this automatically without being told.

**Why this is better than TF-IDF:**

| | TF-IDF | Embedding Layer |
|---|---|---|
| Representation | Single frequency score | 64-dimensional learned vector |
| Similar words ("happy", "joyful") | Treated as completely unrelated | Pushed close together in vector space |
| Fixed or learned | Fixed — never changes | Learned — improves every training step |
| Semantic understanding | ❌ None | ✅ Yes |
| Part of the model | ❌ No — outside the model | ✅ Yes — first layer, trains with everything else |

**Type of Embedding used:** I used a **learned-from-scratch embedding** — the vectors start random and are trained entirely on my emotion dataset. An alternative would be pre-trained embeddings like **GloVe** or **Word2Vec**, which already know word relationships from training on billions of words. Using GloVe would be a potential future improvement.

---
## New Concept 3 — SpatialDropout1D

### The Concept

`SpatialDropout1D` is a regularisation technique placed right after the Embedding layer. It is a specialised version of Dropout designed specifically for sequence data.

**Why I chose it:** Regular Dropout randomly zeros out individual values. For sequence data, this is too granular — embedding features are correlated across all word positions. SpatialDropout1D drops entire channels across the whole sequence, which is more effective for text.

### The Implementation

```python
SpatialDropout1D(0.3)
```

### The Learning — How It Works Under the Hood

**What `0.3` means:** It does NOT remove the value 0.3. It means **30% of the 64 embedding channels are randomly zeroed out** during each training step. With 64 dimensions, approximately 19 channels are dropped.

**Regular Dropout vs SpatialDropout1D:**

Regular Dropout zeros out random individual values scattered across positions:
```
Word 1: [0.23,  0.00,  0.87,  0.04,  0.00, ...]  ← random individual zeros
Word 2: [0.00,  0.12,  0.34,  0.00,  0.21, ...]  ← different random zeros
```

SpatialDropout1D zeros out entire channels across the WHOLE sequence:
```
Word 1: [0.23,  0.00,  0.87,  0.04,  0.00, ...]  ← channel 2 and 5 dropped
Word 2: [0.45,  0.00,  0.34,  0.78,  0.00, ...]  ← SAME channels 2 and 5 dropped
```

This forces the model to not rely on any single embedding dimension, making the learned representations more robust and reducing overfitting.

**Regularisation techniques in my model:**

| Technique | Where | What it drops |
|---|---|---|
| `SpatialDropout1D(0.3)` | After Embedding | 30% of embedding channels across whole sequence |
| `dropout=0.2` inside LSTM | Input connections | 20% of input values at each step |
| `recurrent_dropout=0.2` inside LSTM | Hidden state connections | 20% of hidden state values between steps |
| `Dropout(0.4)` | After BiLSTM | 40% of the 128 BiLSTM output values |
| `EarlyStopping` | During training | Stops training before overfitting gets bad |

---
## New Concept 4 — LSTM (Long Short-Term Memory)

### The Concept

LSTM is a type of Recurrent Neural Network designed to process text **one word at a time** while maintaining a memory of what it has already read.

**Why I chose it:** Dense layers have no concept of word order. LSTMs read sequences step by step with memory, meaning "not happy" is processed differently from "happy not" — which is critical for emotion detection.

### The Implementation

```python
Bidirectional(LSTM(64, dropout=0.2, recurrent_dropout=0.2))
```

### The Learning — How It Works Under the Hood

**The problem with basic RNNs:** Before LSTMs, simple RNNs attempted to process sequences but suffered from the **vanishing gradient problem**. During backpropagation, the error signal shrinks as it travels backwards through many time steps:
```
After 10 steps: gradient × 0.9^10 = 0.35
After 50 steps: gradient × 0.9^50 = 0.005  ← nearly zero, model forgets early words
```

**How LSTM solves this — the Cell State:** LSTM introduces a second memory vector called the **cell state** (`C_t`) alongside the **hidden state** (`h_t`):
- **Hidden state `h_t`** — short-term working memory, changes at every step
- **Cell state `C_t`** — long-term memory highway, only updated when the gates decide

The cell state uses **additive** updates instead of multiplicative ones, which allows gradients to flow backwards without vanishing.

**The Three Gates:**

| Gate | What it does |
|---|---|
| **Forget gate** | Decides what old information to erase from memory. Sigmoid output 0–1: 0 = forget, 1 = keep |
| **Input gate** | Decides what new information from the current word to add to memory |
| **Output gate** | Decides what part of the cell state to pass forward as the hidden state |

The cell state update equation:
```
C_t = (forget_gate × C_(t-1))  +  (input_gate × new_candidate_values)
       ↑ old memory scaled          ↑ new information added
       by how much to forget         based on how much to update
```

This additive operation is why gradients do not vanish — information can flow through the cell state highway across 50 steps without shrinking.

**`LSTM(64)`** — 64 LSTM units. Each unit has its own forget gate, input gate, and output gate. The hidden state and cell state are both 64-dimensional vectors.

**`dropout=0.2`** — randomly drops 20% of the INPUT connections going into the LSTM at each step.

**`recurrent_dropout=0.2`** — randomly drops 20% of the HIDDEN STATE connections between steps. Uses the same dropout mask for all time steps within one sequence (Gal & Ghahramani method) — the model cannot route around the dropout by relying on future steps.

**Visualising the flow:**
```
"I"     → [LSTM cell] → hidden state 1
"am"    → [LSTM cell] → hidden state 2
"not"   → [LSTM cell] → hidden state 3  ← "not" stored in memory
"happy" → [LSTM cell] → hidden state 4  ← knows "not" came before "happy"
...
final hidden state → Dense → emotion prediction
```

---
## New Concept 5 — Bidirectional LSTM

### The Concept

A Bidirectional LSTM runs TWO LSTM layers on the same sequence simultaneously — one forward and one backward — then concatenates their outputs.

**Why I chose it:** A standard LSTM only reads left to right. For emotion detection, the end of a sentence often reframes its beginning. Bidirectional reading gives every word context from both sides simultaneously.

### The Implementation

```python
Bidirectional(LSTM(64, dropout=0.2, recurrent_dropout=0.2))
```

### The Learning — How It Works Under the Hood

**Forward LSTM** reads left → right:
```
"I" → "did" → "not" → "feel" → "happy" → "at" → "all" → "today"
 h1     h2      h3      h4       h5        h6      h7      h8_forward
```

**Backward LSTM** reads right → left:
```
"today" → "all" → "at" → "happy" → "feel" → "not" → "did" → "I"
  h1        h2      h3     h4        h5        h6      h7     h8_backward
```

Both LSTMs have **completely separate weight matrices** — they learn independently.

**Concatenation:** Their final hidden states are joined together:
```
output = [h8_forward, h8_backward] → 128-dimensional vector (64 + 64)
```

**Why 128 connections to Dense(6):**
Because BiLSTM outputs 128 numbers (not 64), the final Dense layer has:
```
128 × 6 = 768 connections
```
If it were a regular one-direction LSTM: 64 × 6 = 384 connections.

**Why reading backwards matters for emotion:**

Consider: *"Despite feeling anxious all morning, I felt pure joy when I saw her."*

- **Forward LSTM** at "anxious" — only knows the negative beginning, not the positive ending
- **Backward LSTM** at "anxious" — already read "joy", "pure", "felt" — knows the positive context coming

By combining both, every word has context from **both sides** simultaneously. This is especially powerful for:
- **Negations** — "not happy" (forward sees "not" before "happy", backward sees "happy" after "not")
- **Contrast sentences** — "I was terrified but then I loved it"
- **Delayed emotion reveals** — emotion revealed at end of sentence

**Output shape comparison:**

| Architecture | Output size |
|---|---|
| `LSTM(64)` — one direction | 64 numbers |
| `Bidirectional(LSTM(64))` | 128 numbers (64 + 64 concatenated) |

---
## New Concept 6 — EarlyStopping with `restore_best_weights`

### The Concept

EarlyStopping is a callback that automatically halts training when the model stops improving on validation data.

**Why I chose it:** Without EarlyStopping, the model trains for all 20 epochs. Later epochs overfit — train accuracy keeps rising while val accuracy flatlines or drops. EarlyStopping catches the best generalisation point automatically.

### The Implementation

```python
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=64,
    validation_data=(X_val, y_val),
    callbacks=[early_stop]
)
```

### The Learning — How It Works Under the Hood

**`monitor='val_loss'`** — watches validation loss after every epoch. We monitor val_loss (not train_loss) because we care about performance on unseen data, not the data the model trained on.

**`patience=3`** — training stops if val_loss fails to improve for 3 consecutive epochs. Without patience, training would stop the first time val_loss didn't improve even slightly — patience gives the model a chance to recover from a temporary bad epoch.

**`restore_best_weights=True`** — when training stops, the model rewinds its weights back to the epoch with the lowest val_loss — not the final epoch. This is critical. Without it, the saved model is from the last (potentially overfit) epoch. With it, the saved model is from the best-generalising point in training.

**My training result:**
Training stopped at epoch 9. The model rewound to approximately epoch 6 where val_loss was at its lowest (~0.286). The final val accuracy was ~91%.

---
## Full Model Architecture — End to End

```python
model = Sequential([
    Embedding(input_dim=10000, output_dim=64, input_length=50),
    SpatialDropout1D(0.3),
    Bidirectional(LSTM(64, dropout=0.2, recurrent_dropout=0.2)),
    Dropout(0.4),
    Dense(num_classes, activation='softmax')
])
```

| Layer | Output Shape | Plain English |
|---|---|---|
| `Embedding` | (batch, 50, 64) | Turns each of the 50 word integers into a 64-number meaningful vector. Output is a 50×64 grid |
| `SpatialDropout1D(0.3)` | (batch, 50, 64) | Randomly zeros 30% of the 64 embedding channels across all 50 positions |
| `Bidirectional(LSTM(64))` | (batch, 128) | Reads the 50-word sequence forwards AND backwards. Outputs 128 numbers (64 each direction) |
| `Dropout(0.4)` | (batch, 128) | Randomly zeros 40% of the 128 BiLSTM output values |
| `Dense(6, softmax)` | (batch, 6) | Converts the 128 numbers into 6 emotion probabilities. Highest = predicted emotion |

**What `softmax` does:** Takes the 6 raw scores from Dense and converts them into probabilities that add up to exactly 1.0:
```
Raw scores:    [2.1,  0.3, -1.2,  0.8, -0.5,  0.1]
After softmax: [0.62, 0.10, 0.02, 0.17, 0.04, 0.05]
                anger  fear  joy  love  sad  surprise
→ Predicted: anger (62% confidence)
```

**Complete data flow:**
```
Raw text: "I did not feel happy at all today"
    ↓ clean_text()
"feel happy today"
    ↓ Tokenizer.texts_to_sequences()
[4, 27, 183]
    ↓ pad_sequences(maxlen=50)
[4, 27, 183, 0, 0, ..., 0]
    ↓ Embedding layer
50 × 64 grid of semantic word vectors
    ↓ SpatialDropout1D(0.3)
30% of channels zeroed
    ↓ Forward LSTM + Backward LSTM
128-dimensional sentence summary
    ↓ Dropout(0.4)
40% of values zeroed
    ↓ Dense(6) + softmax
[0.62, 0.10, 0.02, 0.17, 0.04, 0.05]
    ↓ np.argmax
"anger"
```

---
## Results — Part A vs Part B

### Training Performance

| Metric | Part A (TF-IDF + Dense) | Part B (BiLSTM) |
|---|---|---|
| Val Accuracy | ~87–90% | ~91% |
| Overfitting | Heavy — diverged after epoch 2 | Mild — much tighter gap |
| Epochs trained | Stopped early ~epoch 6–7 | Stopped early at epoch 9 |
| Train accuracy at stop | ~93% | ~97% |
| Val accuracy at stop | ~87–90% | ~91% |

### Why Part B Performed Better

1. **Embedding replaced TF-IDF** — words now carry semantic meaning. "happy" and "joyful" are recognised as similar. TF-IDF treated them as completely unrelated.

2. **BiLSTM replaced Dense** — the model now reads the sentence word by word with memory, preserving word order. "not happy" is processed differently from "happy not".

3. **Bidirectional context** — every word has context from both directions simultaneously. Negations, contrasts, and delayed emotion reveals are all captured.

4. **Better regularisation** — SpatialDropout1D, recurrent_dropout, and EarlyStopping together produced a much tighter train/val gap than Part A.

### Pipeline Differences

| Step | Part A (Baseline) | Part B (BiLSTM) |
|---|---|---|
| Vectorization | `TfidfVectorizer` — outside model | `Tokenizer` + `Embedding` — inside model |
| Text → numbers | TF-IDF sparse matrix | Integer sequences + padding |
| Word order | ❌ Lost | ✅ Preserved |
| Semantic meaning | ❌ None | ✅ Learned during training |
| Architecture | Dense → Dense → Dense | Embedding → BiLSTM → Dense |
| Handles negation | ❌ No | ✅ Yes |
| Overfitting | Heavy | Mild |
| Val Accuracy | ~87–90% | ~91% |

---
## Key Vocabulary — Quick Reference

| Term | Definition |
|---|---|
| **TF-IDF** | Converts words to frequency-based scores. Loses word order. |
| **Bag-of-words** | Treating a sentence as a collection of words with no order |
| **Dense layer** | Basic layer where every neuron connects to every neuron in the next layer. No sequence awareness. |
| **Tokenizer** | Builds a dictionary mapping every word to a unique integer |
| **pad_sequences** | Makes all sequences the same length by adding zeros to short ones |
| **OOV token** | Out-Of-Vocabulary — placeholder for words never seen in training |
| **Embedding** | A learned lookup table mapping each word integer to a rich vector of numbers |
| **Vector** | A list of numbers representing something — a word, a sentence, a hidden state |
| **Sparse vector** | A vector mostly filled with zeros (like TF-IDF output) |
| **Dense vector** | A vector where most values are non-zero and meaningful (like Embedding output) |
| **RNN** | Recurrent Neural Network — reads sequences one step at a time |
| **Hidden state** | The LSTM's memory — a vector summarising everything read so far |
| **Cell state** | LSTM's long-term memory highway — allows gradients to flow without vanishing |
| **Vanishing gradient** | Training problem where error signals shrink to near zero in long sequences |
| **Forget gate** | LSTM component that decides what to erase from memory |
| **Input gate** | LSTM component that decides what new information to store |
| **Output gate** | LSTM component that decides what to pass forward as the hidden state |
| **Sigmoid** | Activation function outputting 0–1, used as an on/off switch in LSTM gates |
| **Bidirectional** | Running two LSTMs — one forward, one backward — combining both outputs |
| **Concatenate** | Joining two vectors end-to-end. BiLSTM joins 64 + 64 = 128 |
| **SpatialDropout1D** | Drops entire embedding channels across the whole sequence (not individual values) |
| **recurrent_dropout** | Dropout applied to hidden state connections inside the LSTM |
| **Dropout** | Randomly switches off a percentage of neurons during training |
| **ReLU** | Activation: if value negative output 0, otherwise keep it |
| **Softmax** | Converts raw scores into probabilities summing to 1.0 — used for multi-class output |
| **EarlyStopping** | Automatically stops training when val_loss stops improving |
| **val_loss** | Loss on validation set — measures how well the model generalises to unseen data |
| **patience** | How many epochs val_loss can fail to improve before EarlyStopping halts training |
| **restore_best_weights** | Rewinds model to its best checkpoint after early stopping |
| **Regularisation** | Any technique used to reduce overfitting — Dropout, SpatialDropout, EarlyStopping |
| **Overfitting / Memorisation** | Model memorises training data instead of learning general patterns |
| **Learned from scratch** | Embedding type where vectors start random and are trained on your dataset only |
| **Pre-trained embedding** | Embedding type (e.g. GloVe, Word2Vec) trained on massive external datasets |